# Train the 3di-men model on GPU

Trains the same model as `ml/threedimen/threedimen/model/` (point-cloud
autoencoder + text encoder), just on a real GPU here instead of your local
CPU -- for when the dataset has grown past what's practical to iterate on
locally (hundreds of real Hunyuan3D samples, not tens of mock ones).

**Before running:**
1. On your machine, export a slim training bundle (this drops the raw
   `.stl`/`.glb` copies, which dominate dataset size for real Hunyuan3D
   output at 60k-225k vertices/mesh -- training never reads them anyway):
   ```bash
   cd ml/threedimen
   3dimen dataset export-training-bundle --root data/threedimen_v1_hunyuan --out bundle.zip
   ```
2. On kaggle.com: **Datasets -> New Dataset -> Upload** that `bundle.zip`
   (Kaggle auto-extracts zips into the dataset), name it anything, create it.
3. In this notebook's right sidebar: **Add Input** -> attach that dataset.
4. Settings panel: **Accelerator -> GPU T4 x2** (or P100/whatever's
   available), **Internet -> On**.
5. **If you're pulling Cap3D data (section 3b) at any real scale, don't run
   this as an interactive Draft Session** -- that's tied to your browser
   tab and can disconnect from inactivity or a network blip, losing the
   run. Use **Save Version -> Save & Run All (Commit)** instead: Kaggle
   runs it on its own infrastructure independent of your tab staying open,
   so you can close the browser entirely and check back later. Section 3b
   also checkpoints its own progress to disk as it goes and is resumable
   across separate commits (see that section for exactly how) -- so even
   an interrupted run isn't wasted.

**Quality expectation, same as always:** hundreds of samples is still far
below what a generative 3D model needs to fully generalize. GPU + more data
than the original ~40-sample proof of concept should visibly improve
results, but this is not Hunyuan3D-scale quality and shouldn't be presented
as such.

## 0. Confirm GPU + find your uploaded bundle

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU visible -- set Accelerator to GPU T4 x2 or P100, then Factory Reset the session."
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")

import glob
manifest_candidates = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert manifest_candidates, (
    "No manifest.json found under /kaggle/input -- did you upload the bundle "
    "zip as a Kaggle Dataset and attach it via 'Add Input'?"
)
DATASET_ROOT = manifest_candidates[0].rsplit("/manifest.json", 1)[0]
print(f"using dataset root: {DATASET_ROOT}")

## 1. Install the small set of packages this needs

In [ ]:
# torch/numpy/scipy are already on Kaggle's base image. Only trimesh isn't.
!pip install -q trimesh
print("installed")

## 2. Model code

Pasted directly (matches `ml/threedimen/threedimen/model/{vocab,networks,chamfer}.py`)
so this notebook doesn't depend on the `3di-men` GitHub repo being public/current.

In [ ]:
from __future__ import annotations
import re
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import torch.nn.functional as F

_TOKEN_RE = re.compile(r"[a-z0-9]+")
PAD = "<pad>"
UNK = "<unk>"

def tokenize(text: str) -> list:
    return _TOKEN_RE.findall(text.lower())

@dataclass
class Vocabulary:
    token_to_id: dict = field(default_factory=dict)

    def __len__(self) -> int:
        return len(self.token_to_id)

    @property
    def pad_id(self) -> int:
        return self.token_to_id[PAD]

    @classmethod
    def build(cls, texts, min_freq: int = 1) -> "Vocabulary":
        counts = {}
        for text in texts:
            for token in tokenize(text):
                counts[token] = counts.get(token, 0) + 1
        tokens = [PAD, UNK] + sorted(tok for tok, c in counts.items() if c >= min_freq)
        return cls(token_to_id={tok: i for i, tok in enumerate(tokens)})

    def encode(self, text: str, max_len: int) -> list:
        unk_id = self.token_to_id[UNK]
        ids = [self.token_to_id.get(tok, unk_id) for tok in tokenize(text)][:max_len]
        ids += [self.pad_id] * (max_len - len(ids))
        return ids

    def to_dict(self) -> dict:
        return {"token_to_id": self.token_to_id}


class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.project = nn.Sequential(nn.Linear(embed_dim, latent_dim), nn.ReLU(), nn.Linear(latent_dim, latent_dim))

    def forward(self, token_ids):
        mask = (token_ids != 0).float().unsqueeze(-1)
        embedded = self.embedding(token_ids) * mask
        pooled = embedded.sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return self.project(pooled)


class PointCloudEncoder(nn.Module):
    def __init__(self, hidden_dim, latent_dim):
        super().__init__()
        self.point_mlp = nn.Sequential(
            nn.Linear(3, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, points):
        per_point = self.point_mlp(points)
        pooled, _ = per_point.max(dim=1)
        return pooled


class PointCloudDecoder(nn.Module):
    def __init__(self, latent_dim, num_points, hidden_dim):
        super().__init__()
        self.num_points = num_points
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, num_points * 3),
        )

    def forward(self, latent):
        return self.mlp(latent).view(-1, self.num_points, 3)


def chamfer_distance(pred, target):
    dist = torch.cdist(pred, target)
    pred_to_target = dist.min(dim=2).values.mean(dim=1)
    target_to_pred = dist.min(dim=1).values.mean(dim=1)
    return (pred_to_target + target_to_pred).mean()

print("model code loaded")

## 3. Load the training bundle

Mirrors `ml/threedimen/threedimen/model/data.py`'s `load_training_pairs`,
standalone (reads the bundle's manifest.json + each sample's
surface.json/.npz directly, without needing the full `threedimen` package).

In [ ]:
import json
import numpy as np
import trimesh
from pathlib import Path

NUM_POINTS = 2048

def load_surface_points(surface_json_path: Path, num_points: int, rng: np.random.Generator):
    npz_path = surface_json_path.with_suffix(".npz")
    arrays = dict(np.load(npz_path))
    mesh = trimesh.Trimesh(vertices=arrays["patch0:vertices"], faces=arrays["patch0:faces"], process=False)
    points, _ = trimesh.sample.sample_surface(mesh, num_points, seed=rng)
    return np.asarray(points)

def load_training_pairs(root: str, num_points: int, seed: int = 0):
    root = Path(root)
    manifest = json.loads((root / "manifest.json").read_text())
    rng = np.random.default_rng(seed)
    pairs = []
    for sample in manifest["samples"]:
        surface_json = root / f"{sample['surface_representation_path']}.json"
        if not surface_json.with_suffix(".npz").exists():
            continue
        points = load_surface_points(surface_json, num_points, rng)
        pairs.append({"prompt": sample["prompt"], "points": points})
    return pairs

pairs = load_training_pairs(DATASET_ROOT, NUM_POINTS)
print(f"loaded {len(pairs)} training pairs")
print("example:", pairs[0]["prompt"])

## 3b. Pull Cap3D data (optional) and train -- all in one cell

Kaggle has its own internet access, so this can pull real-world (caption,
mesh) pairs from [Cap3D](https://huggingface.co/datasets/tiange/Cap3D)
straight into this session -- no local download / bundle re-export / re-
upload round-trip needed for this specific data source. Mirrors
`ml/threedimen/threedimen/external/cap3d.py`'s logic standalone, same as
the model code above.

**This cell does the Cap3D pull AND the training loop, in that order,
as one single run.** Running the download and training as two separate
cells left room for training to silently get skipped (or run on stale
`pairs`) if you only executed one of them -- merging them means a single
"run this cell" always does both, in the right order, on the same data.

**Live progress bars for both phases** (via `tqdm`): a download bar
showing chunks processed / added / license-skipped / not-found / rate, and
a training bar showing the current epoch's recon/align loss -- so you can
see at a glance how far either phase has gotten without reading through
scrollback.

**On "the entire corpus":** no single Kaggle session can pull, license-
check, and process all of Cap3D/Objaverse -- that's a network throughput
limit (hours just to download at any realistic rate), not a code-quality
one. What actually gets you there is running this notebook repeatedly with
progress persisted between runs, each one picking up past where the last
left off. That's what the download half of this cell does:

- **Streaming, chunked pipeline.** Candidates are walked through in small
  batches of `CAP3D_CHUNK_SIZE` (license-checked *and* downloaded *and*
  processed together, one chunk at a time) instead of license-checking a
  huge batch of candidates upfront before any downloading starts. This
  matters beyond just responsiveness: `objaverse.load_annotations()` fails
  the whole batch it's given if even one UID in it is bad, and real runs
  have seen a large fraction of UIDs come back "not found" -- so a big
  batch with many bad UIDs scattered through it can force the
  divide-and-conquer fallback to recurse almost down to individual calls
  anyway, which is likely why a large upfront batch made earlier runs slow
  to show any progress at all. Small chunks bound that worst case tightly
  (at most `CAP3D_CHUNK_SIZE` calls per chunk).
- **Disk-safe.** Each chunk's objects are downloaded, immediately
  converted to point clouds and saved (a few KB each), and the raw
  downloaded mesh is deleted right after. Disk usage stays flat regardless
  of how many total objects get processed.
- **Resumable across separate Kaggle sessions.** Progress (which UIDs are
  done + their point clouds) is written to `/kaggle/working/cap3d_progress/`
  as it goes. After a run, its **Output** contains that folder. To continue
  next time instead of starting over: **Add Input -> search your own
  notebooks -> this notebook** (Kaggle attaches its latest saved version's
  output under `/kaggle/input/...`) -- the cell below detects and resumes
  from it automatically, skipping UIDs already collected.
- **A time budget** (`CAP3D_TIME_BUDGET_MINUTES`) stops the pull with
  progress already saved, leaving headroom in the session for training --
  rather than running out the clock mid-download with nothing usable
  flushed to disk.

`CAP3D_TARGET_NEW = None` (the default) means "don't stop at some fixed
count -- keep working through the corpus, chunk by chunk, until the time
budget runs out." Combined with resuming across runs, repeated commits
genuinely work through the whole corpus over time instead of capping at an
arbitrary number each run. Set it to an integer instead if you want a given
run to stop earlier than its time budget would.

**License note** (same as the standalone module): Cap3D's captions are
ODC-By, but the underlying Objaverse assets carry mixed, per-object
licenses -- this does a best-effort permissive-license filter, not a legal
guarantee.

Set `USE_CAP3D = False` below to skip the pull and train on the uploaded
bundle only.

In [ ]:
import glob
import json
import math
import os
import shutil
import time
from pathlib import Path

from tqdm.auto import tqdm

USE_CAP3D = True
# None = don't cap this run at a fixed count -- keep working through the
# corpus, chunk by chunk, until CAP3D_TIME_BUDGET_MINUTES runs out. Set an
# integer instead to stop a given run earlier.
CAP3D_TARGET_NEW = None
CAP3D_SEED = 0
# Kaggle is Linux (fork-based multiprocessing) -- unlike testing this
# locally on macOS (spawn-based, which crashed at processes=4 without a
# `if __name__ == "__main__":` guard), this can safely parallelize
# downloads here. If you hit multiprocessing errors, drop this back to 1.
CAP3D_DOWNLOAD_PROCESSES = 8
# Small on purpose: each chunk is license-checked, downloaded, and
# processed as one unit, so a smaller chunk means faster/more frequent
# progress updates AND a tighter worst-case bound on the divide-and-
# conquer license-check fallback (see markdown above for why a big
# upfront batch was likely the real cause of slow early progress).
CAP3D_CHUNK_SIZE = 50
CAP3D_TIME_BUDGET_MINUTES = 300  # stop with progress saved, leaving time for training

PROGRESS_DIR = Path("/kaggle/working/cap3d_progress")
PROGRESS_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = PROGRESS_DIR / "manifest.jsonl"

!pip install -q objaverse huggingface_hub tqdm

cap3d_pairs = []
if USE_CAP3D:
    import random
    import pandas as pd
    from huggingface_hub import hf_hub_download
    import objaverse

    _PERMISSIVE_MARKERS = ("by", "cc0", "public-domain", "by-sa")

    def _is_permissive(lic: str) -> bool:
        lic = lic.lower()
        return any(m in lic for m in _PERMISSIVE_MARKERS)

    def _load_annotations_robust(uids):
        # objaverse.load_annotations() raises for the WHOLE batch if even
        # one requested UID can't be resolved. Divide-and-conquer, not flat
        # one-by-one -- and with CAP3D_CHUNK_SIZE small, even the worst
        # case (every sub-split contains a bad uid, forcing recursion to
        # singletons) costs at most ~2x CAP3D_CHUNK_SIZE calls, not the
        # much larger blowup a huge upfront batch could hit.
        if not uids:
            return {}
        try:
            return objaverse.load_annotations(uids)
        except Exception:
            pass
        if len(uids) == 1:
            return {}  # this one UID genuinely can't be resolved -- skip it, not the whole batch
        mid = len(uids) // 2
        result = _load_annotations_robust(uids[:mid])
        result.update(_load_annotations_robust(uids[mid:]))
        return result

    def _recenter_and_normalize(mesh, target_diagonal=1.0):
        # MUST match geometry/ingest.py's normalization exactly -- mixing
        # un-normalized points into the same training batch as the bundle's
        # normalized points previously blew up recon/align loss into the
        # thousands/billions.
        bounds = mesh.bounds
        center = (bounds[0] + bounds[1]) / 2
        diagonal = float(np.linalg.norm(bounds[1] - bounds[0]))
        mesh = mesh.copy()
        mesh.apply_translation(-center)
        if diagonal > 1e-9:
            mesh.apply_scale(target_diagonal / diagonal)
        return mesh

    # ---- Resume from a prior run's Output, if attached as an input this
    # session (Add Input -> search your own notebooks -> this notebook).
    # Without this, every run starts from zero.
    prior_progress = glob.glob("/kaggle/input/**/cap3d_progress", recursive=True)
    if prior_progress and not MANIFEST_PATH.exists():
        print(f"resuming from prior progress: {prior_progress[0]}")
        shutil.copytree(prior_progress[0], PROGRESS_DIR, dirs_exist_ok=True)

    done_uids = set()
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH) as f:
            for line in f:
                done_uids.add(json.loads(line)["uid"])
    print(f"{len(done_uids)} samples already collected from prior runs")

    captions_path = hf_hub_download(
        repo_id="tiange/Cap3D", filename="Cap3D_automated_Objaverse_full.csv", repo_type="dataset"
    )
    captions_df = pd.read_csv(captions_path, header=None, names=["uid", "caption"])
    caption_by_uid = dict(zip(captions_df["uid"], captions_df["caption"]))
    TOTAL_CORPUS_SIZE = len(captions_df)
    print(f"Cap3D corpus: {TOTAL_CORPUS_SIZE} total captioned objects")

    rng = random.Random(CAP3D_SEED)
    candidate_uids = captions_df["uid"].tolist()
    rng.shuffle(candidate_uids)
    candidate_uids = [u for u in candidate_uids if u not in done_uids]
    print(f"{len(candidate_uids)} not-yet-attempted candidates remain this run")

    target = CAP3D_TARGET_NEW if CAP3D_TARGET_NEW is not None else len(candidate_uids)
    total_chunks = math.ceil(min(target, len(candidate_uids)) / CAP3D_CHUNK_SIZE) if candidate_uids else 0

    rng_np = np.random.default_rng(CAP3D_SEED)
    t0 = time.monotonic()
    added_this_run = 0
    attempted_this_run = 0
    license_skipped_total = 0
    not_found_total = 0
    stopped_early = False
    chunk_num = 0

    download_bar = tqdm(total=total_chunks, desc="Cap3D download", unit="chunk")
    with open(MANIFEST_PATH, "a") as manifest_f:
        for chunk_start in range(0, len(candidate_uids), CAP3D_CHUNK_SIZE):
            if added_this_run >= target:
                tqdm.write(f"reached this run's target of {target} new samples")
                break
            elapsed_min = (time.monotonic() - t0) / 60
            if elapsed_min > CAP3D_TIME_BUDGET_MINUTES:
                tqdm.write(f"time budget ({CAP3D_TIME_BUDGET_MINUTES} min) reached, stopping with progress saved")
                stopped_early = True
                break

            chunk_num += 1
            raw_chunk = candidate_uids[chunk_start: chunk_start + CAP3D_CHUNK_SIZE]
            attempted_this_run += len(raw_chunk)

            # license-check just THIS chunk (cheap: at most CAP3D_CHUNK_SIZE candidates)
            annotations = _load_annotations_robust(raw_chunk)
            permissive_chunk = []
            license_skipped = 0
            for uid in raw_chunk:
                lic = str((annotations.get(uid) or {}).get("license") or "")
                if lic and not _is_permissive(lic):
                    license_skipped += 1
                    continue
                permissive_chunk.append(uid)
            license_skipped_total += license_skipped

            downloaded = objaverse.load_objects(uids=permissive_chunk, download_processes=CAP3D_DOWNLOAD_PROCESSES)

            chunk_added = 0
            not_found = 0
            for uid in permissive_chunk:
                mesh_path = downloaded.get(uid)
                if mesh_path is None:
                    not_found += 1
                    continue
                try:
                    mesh = trimesh.load(mesh_path, force="mesh")
                    mesh = _recenter_and_normalize(mesh)
                    points, _ = trimesh.sample.sample_surface(mesh, NUM_POINTS, seed=rng_np)
                    np.savez_compressed(PROGRESS_DIR / f"{uid}.npz", points=np.asarray(points))
                    manifest_f.write(json.dumps({"uid": uid, "prompt": str(caption_by_uid[uid])}) + "\n")
                    manifest_f.flush()
                    chunk_added += 1
                except Exception as e:
                    tqdm.write(f"  skipping {uid}: {e}")
                finally:
                    # process-then-delete: disk stays bounded to one chunk
                    # regardless of total samples accumulated across all runs
                    if mesh_path is not None and os.path.exists(mesh_path):
                        os.remove(mesh_path)

            added_this_run += chunk_added
            not_found_total += not_found
            elapsed_min = (time.monotonic() - t0) / 60
            rate = added_this_run / elapsed_min if elapsed_min > 0.01 else 0.0
            total_collected = len(done_uids) + added_this_run
            pct_of_corpus = 100 * total_collected / TOTAL_CORPUS_SIZE

            download_bar.update(1)
            download_bar.set_postfix(
                added=added_this_run,
                lic_skip=license_skipped_total,
                not_found=not_found_total,
                pct_corpus=f"{pct_of_corpus:.2f}%",
                rate_per_min=f"{rate:.1f}",
            )
            eta_str = ""
            if CAP3D_TARGET_NEW is not None and rate > 0:
                eta_min = max(target - added_this_run, 0) / rate
                eta_str = f" | ETA-to-target~{eta_min:.0f}min"
            tqdm.write(
                f"[chunk {chunk_num}/{total_chunks}] this chunk: {len(raw_chunk)} candidates, "
                f"{license_skipped} license-skipped, {not_found} not-found, {chunk_added} added || "
                f"run totals: {added_this_run} added, {attempted_this_run} attempted || "
                f"ALL-TIME: {total_collected} collected ({pct_of_corpus:.3f}% of {TOTAL_CORPUS_SIZE}-object corpus) || "
                f"{elapsed_min:.1f}min elapsed, {rate:.1f}/min{eta_str}"
            )
    download_bar.close()

    print(
        f"\nrun finished: added {added_this_run} new samples "
        f"(attempted {attempted_this_run}, {license_skipped_total} license-skipped, {not_found_total} not-found)"
        + (" -- stopped early on time budget" if stopped_early else " -- exhausted target/candidates")
    )

    # Load ALL cumulative progress (this run + every prior resumed run),
    # not just what was just added, so training sees everything collected so far.
    with open(MANIFEST_PATH) as f:
        for line in f:
            entry = json.loads(line)
            npz_path = PROGRESS_DIR / f"{entry['uid']}.npz"
            if npz_path.exists():
                cap3d_pairs.append({"prompt": entry["prompt"], "points": np.load(npz_path)["points"]})

    print(f"total cumulative Cap3D samples available for training: {len(cap3d_pairs)} "
          f"({100 * len(cap3d_pairs) / TOTAL_CORPUS_SIZE:.3f}% of the full corpus)")

pairs = pairs + cap3d_pairs
print(f"total training pairs (bundle + Cap3D): {len(pairs)}")


# =====================================================================
# Train -- merged into this same cell so a single run always does both
# the Cap3D pull above and this training loop, on the same `pairs`.
# =====================================================================
from torch.utils.data import Dataset, DataLoader

EMBED_DIM = 128
HIDDEN_DIM = 256
LATENT_DIM = 128
MAX_LEN = 32
EPOCHS = 300
BATCH_SIZE = 32
LR = 1e-3
ALIGN_WEIGHT = 1.0
SEED = 0

device = torch.device("cuda")
torch.manual_seed(SEED)

vocab = Vocabulary.build([p["prompt"] for p in pairs])
print(f"vocab size: {len(vocab)}")

class PairDataset(Dataset):
    def __init__(self, pairs, vocab, max_len):
        self.pairs = pairs
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        tokens = torch.tensor(self.vocab.encode(pair["prompt"], self.max_len), dtype=torch.long)
        points = torch.tensor(pair["points"], dtype=torch.float32)
        return tokens, points

dataset = PairDataset(pairs, vocab, MAX_LEN)
loader = DataLoader(dataset, batch_size=min(BATCH_SIZE, len(dataset)), shuffle=True, pin_memory=True)

text_encoder = TextEncoder(len(vocab), EMBED_DIM, LATENT_DIM, pad_idx=vocab.pad_id).to(device)
shape_encoder = PointCloudEncoder(HIDDEN_DIM, LATENT_DIM).to(device)
shape_decoder = PointCloudDecoder(LATENT_DIM, NUM_POINTS, HIDDEN_DIM).to(device)

params = list(text_encoder.parameters()) + list(shape_encoder.parameters()) + list(shape_decoder.parameters())
optimizer = torch.optim.Adam(params, lr=LR)

t0 = time.perf_counter()
train_bar = tqdm(range(EPOCHS), desc="Training", unit="epoch")
for epoch in train_bar:
    epoch_recon = epoch_align = 0.0
    n_batches = 0
    for tokens, points in loader:
        tokens = tokens.to(device, non_blocking=True)
        points = points.to(device, non_blocking=True)
        optimizer.zero_grad()

        shape_latent = shape_encoder(points)
        recon = shape_decoder(shape_latent)
        recon_loss = chamfer_distance(recon, points)

        text_latent = text_encoder(tokens)
        align_loss = F.mse_loss(text_latent, shape_latent.detach())

        loss = recon_loss + ALIGN_WEIGHT * align_loss
        loss.backward()
        optimizer.step()

        epoch_recon += recon_loss.item()
        epoch_align += align_loss.item()
        n_batches += 1

    mean_recon = epoch_recon / n_batches
    mean_align = epoch_align / n_batches
    train_bar.set_postfix(recon=f"{mean_recon:.5f}", align=f"{mean_align:.5f}")

    if epoch % 20 == 0 or epoch == EPOCHS - 1:
        tqdm.write(f"epoch {epoch:4d}  recon={mean_recon:.5f}  align={mean_align:.5f}  "
                   f"elapsed={time.perf_counter()-t0:.1f}s")

print(f"training finished in {time.perf_counter()-t0:.1f}s")

## 4. Save the checkpoint (CPU-portable -- loadable on any machine, GPU or not)

In [ ]:
from dataclasses import asdict

config = {
    "num_points": NUM_POINTS, "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM,
    "latent_dim": LATENT_DIM, "max_len": MAX_LEN, "epochs": EPOCHS,
    "batch_size": BATCH_SIZE, "lr": LR, "align_weight": ALIGN_WEIGHT, "seed": SEED,
}

checkpoint_path = "/kaggle/working/threedimen_v2.pt"
torch.save(
    {
        "text_encoder": text_encoder.state_dict(),
        "shape_encoder": shape_encoder.state_dict(),
        "shape_decoder": shape_decoder.state_dict(),
        "vocab": vocab.to_dict(),
        "config": config,
    },
    checkpoint_path,
)
print(f"saved {checkpoint_path}")

## 5. Sanity-check generation before downloading anything

In [ ]:
def generate_mesh(prompt: str):
    with torch.no_grad():
        tokens = torch.tensor([vocab.encode(prompt, MAX_LEN)], dtype=torch.long, device=device)
        latent = text_encoder(tokens)
        points = shape_decoder(latent)[0].cpu().numpy()
    return trimesh.Trimesh(vertices=points).convex_hull

for test_prompt in ["a compact aluminum bracket with mounting holes", "a heavy-duty steel pump housing"]:
    mesh = generate_mesh(test_prompt)
    print(f"{test_prompt!r} -> faces={len(mesh.faces)} watertight={mesh.is_watertight}")

## 6. Getting the checkpoint back to your machine

Left sidebar file browser -> `/kaggle/working/` -> `...` menu on
`threedimen_v2.pt` -> **Download**. (Or **Save Version -> Save & Run All**,
then grab it from the notebook's **Output** tab if you want it to persist
past this session.)

Then either:
- **Use it locally**: `3dimen model generate "..." --checkpoint threedimen_v2.pt --out out.stl`
- **Serve it live**: upload it as a new Kaggle Dataset (or a new version of
  the existing one) and point `threedimen_api_server.ipynb` at it, same as
  the original CPU-trained checkpoint -- nothing about the serving notebook
  needs to change, a GPU-trained checkpoint loads and runs on CPU inference
  identically (this model is tiny either way).